# ReBRAC Stage E (a) — `crosscomp-1000` critic-penalty-off cross-dataset 二次验证

**目的**：把 Stage D 在 worldcomp 上做的 critic-penalty-off probe（[report §7.13](../docs/rebrac_experiment_report.md)）扩展到 crosscomp-1000，回答两个问题：

1. **Cross-dataset Finding 1**：Stage D probe 显示 worldcomp 上 `(β1=4.0, β2=0)` 同 seeds (42/43) 比 Phase 1 主 finalist 掉 5pp，且 mean_target_q 跳 +46%。这一 "critic penalty 对 Q 稳定性大且必要、对 mean success 小但非零" 的故事是否在 crosscomp 上也成立？
2. **Stage E 收口**：是否需要保留 dual penalty 这一项作为 ReBRAC 主线必需，还是可以在 typical regime 下用单 actor penalty 替代？

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 | `4.0` | Stage C 锁定的主 finalist β1 |
| **β2** | **`0.0`** | 关键操纵：critic penalty off |
| dataset | `crosscomp-1000` | Stage C 主对照点（0.902 ± 0.021） |
| seeds | `42 43 44 45 46` | 与 Stage C 完全对齐 |
| TRAIN_EPOCHS | `64` | 与 Stage B/C/D 全线一致 |
| 轨道 | deployable | actor + critic 都用 deployable obs |
| val / test manifest | 40 / 100 | 复用 Stage C `benchmarks/offline_rebrac_screen/` |
| 算力 | ≈ Stage C 单 finalist 的一半（仅 1 dataset） | |

**输出树**（与 Stage B/C/D 完全隔离）：
- `checkpoints/offline/rebrac/stage_e_critic_penalty_off/`
- `results/offline/rebrac/stage_e_critic_penalty_off/`

**对照基线**（与 Stage E 报告章节硬编码一致）：

| 协议 | dataset | mean | std | 来源 |
| --- | --- | --- | --- | --- |
| ReBRAC `(β1=4.0, β2=2.0)` | crosscomp-1000 | 0.902 | 0.021 | Stage C |
| ReBRAC `(β1=4.0, β2=1.0)` | crosscomp-1000 | — | — | Stage B 3-seed only |
| ReBRAC `(β1=4.0, β2=0)` worldcomp probe | worldcomp-1000 | 0.910 | 0.014 | Stage D probe |
| ReBRAC `(β1=4.0, β2=0)` 当前实验 | crosscomp-1000 | ?? | ?? | Stage E (a) |

**判定区间**（与 Stage D probe 对齐，方便 cross-dataset 对照）：

| 情形 | mean_test_success | 解读 |
| --- | --- | --- |
| **A** 几乎一致 | `0.882 ~ 0.922`（Stage C ± 2pp） | dual penalty 在 crosscomp 上也几乎可省 → Finding 1 跨 dataset 成立；Stage E 收口 |
| **B** 略有下降 | `0.85 ~ 0.88` | 与 worldcomp probe 量级相同；critic penalty 对 mean 贡献 "小但非零" 跨 dataset 成立 |
| **C** 显著下降 | `< 0.85` | crosscomp 与 worldcomp 表现不同；critic penalty 在 crosscomp 上贡献更大；需要重写 Finding 1 |

**核心诊断量**：与 worldcomp probe 一致——`mean_target_q` 是否同样有 +30%~+50% 的跳升？这才是 critic penalty 真正的作用渠道（target Q bootstrap 抑制 Q 高估）。

## 0. 环境 sanity check

In [ ]:
!lscpu | head -10
print()
!nvidia-smi

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   85

Wed Apr 29 15:21:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-U

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 1. 环境配置

复用 `scripts/run_offline_rebrac_screen.sh`（Stage B/C 共用 driver），通过 env-var 改成单 config：

- `ACTOR_PENALTY_COEFS="4.0"`、`CRITIC_PENALTY_COEFS="0.0"`；
- `DATASET_POLICY=crosscomp`、`DATASET_EPISODES="1000"`；
- `SEEDS="42 43 44 45 46"`；
- 输出根目录隔离到 `stage_e_critic_penalty_off/`；
- `MANIFEST_ROOT` 用 driver 默认值 `benchmarks/offline_rebrac_screen`，与 Stage C 完全复用 manifests（test=100 已存在）。

In [ ]:
import os

# —— 通用（与 Stage B/C/D 全线一致）——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— Stage E (a) 关键操纵：β2=0 ——
os.environ["ACTOR_PENALTY_COEFS"]  = "4.0"
os.environ["CRITIC_PENALTY_COEFS"] = "0.0"

# —— 协议（与 Stage C 主 finalist 完全对齐）——
os.environ["DATASET_POLICY"]          = "crosscomp"
os.environ["DATASET_EPISODES"]        = "1000"
os.environ["SEEDS"]                   = "42 43 44 45 46"
os.environ["TRAIN_EPOCHS"]            = "64"
os.environ["CHECKPOINT_EVERY_EPOCHS"] = "8"
os.environ["VAL_MANIFEST_EPISODES"]   = "40"
os.environ["TEST_MANIFEST_EPISODES"]  = "100"

# —— Manifest 复用 Stage C（driver 默认 benchmarks/offline_rebrac_screen）——
# 不覆盖 MANIFEST_ROOT；Stage C 时 test_100 / val_40 已经生成。

# —— 输出根目录隔离（不污染 Stage B/C/D）——
os.environ["CHECKPOINT_ROOT"] = "checkpoints/offline/rebrac/stage_e_critic_penalty_off"
os.environ["RESULTS_ROOT"]    = "results/offline/rebrac/stage_e_critic_penalty_off"
os.environ["SUMMARY_ROOT"]    = "results/offline/rebrac/stage_e_critic_penalty_off/summaries"

# 其余默认（BENCHMARK_KEY=single_u10_cross_tgt15, PROBE_LAYOUT=s0,
# HISTORY_LENGTH=4, TASK_GEOMETRY=cross_stream, TARGET_SPEED=1.5,
# OBJECTIVE=efficiency_v2, SAMPLING_MODE=shuffle_no_replacement,
# BATCH_SIZE=256, USE_ASYMMETRIC_CRITIC=0 即 deployable 轨道）

## 2. Manifest / 离线数据复用 sanity check

Stage C 时已经生成：
- `benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json`
- `benchmarks/offline_rebrac_screen/test_100/single_u10_cross_tgt15.json`
- `offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz`

如有缺失，driver 会自动重建（`MODE=manifests` / `MODE=collect`）。

In [ ]:
import pathlib

manifest_val  = pathlib.Path("benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json")
manifest_test = pathlib.Path("benchmarks/offline_rebrac_screen/test_100/single_u10_cross_tgt15.json")
dataset       = pathlib.Path("offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz")

for label, p in [("val manifest", manifest_val), ("test manifest", manifest_test), ("offline dataset", dataset)]:
    if p.exists():
        print(f"[reuse] {label}: {p}")
    else:
        print(f"[warn] {label} 缺失：{p}（driver 会自动重建）")

# 显式触发 manifests + collect（如果已存在会被 skip）
for mode in ["manifests", "collect"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_screen.sh

[reuse] val manifest: benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json
[reuse] test manifest: benchmarks/offline_rebrac_screen/test_100/single_u10_cross_tgt15.json
[reuse] offline dataset: offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz
[skip] manifest exists: benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json
[skip] manifest exists: benchmarks/offline_rebrac_screen/test_100/single_u10_cross_tgt15.json
[skip] dataset exists: offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz


## 3. 全流程：train → validate → select → test → summarize

5 个 train run（5 seeds × 1 finalist × 64 epoch），每个产出 8 个 ckpt（每 8 epoch），每个 ckpt 在 val=40 上跑一次。selection 按 `success_rate → return → -safety_cost → -time` 选最佳 ckpt，最佳 ckpt 在 test=100 上重跑作为该 (seed) 的最终成绩。

In [ ]:
for mode in ["train", "validate", "test", "summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_screen.sh

[skip] manifest exists: benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json
[skip] manifest exists: benchmarks/offline_rebrac_screen/test_100/single_u10_cross_tgt15.json
[skip] dataset exists: offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz

[cmd] python3 -m scripts.train_offline --algo rebrac --offline-data offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz --flow wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy --manifest benchmarks/offline_rebrac_screen/val_40/single_u10_cross_tgt15.json --probe-layout s0 --history-length 4 --task-geometry cross_stream --target-speed 1.5 --objective efficiency_v2 --sampling-mode shuffle_no_replacement --num-epochs 64 --total-steps 38208 --batch-size 256 --hidden-dim 256 --num-hidden-layers 3 --actor-lr 3e-4 --critic-lr 3e-4 --gamma 0.99 --tau 0.005 --actor-penalty-coef 4.0 --critic-penalty-coef 0.0 --policy-noise 0.2 --noise-clip 0.5 --poli

## 4. 主结果：5-seed test overview

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path("results/offline/rebrac/stage_e_critic_penalty_off")
DATASET_NAME = "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
PAIR_TAG     = "actorb_4p0__criticb_0p0"
SEEDS        = os.environ["SEEDS"].split()


def load_test_per_seed() -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        path = RESULTS_ROOT / DATASET_NAME / PAIR_TAG / "test" / f"seed_{seed}.json"
        if not path.exists():
            print(f"[warn] missing test json: {path}")
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "success_rate": payload["eval_success_rate"],
            "return": payload["eval_return"],
            "safety_cost": payload["eval_safety_cost"],
            "time_s": payload["eval_time_s"],
            "path_efficiency": payload.get("eval_path_efficiency"),
        })
    return pd.DataFrame(rows)


per_seed = load_test_per_seed()
print("[per-seed test results — Stage E (a) crosscomp-1000 (β1=4.0, β2=0)]")
print(per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if not per_seed.empty:
    print()
    print("[summary]")
    print(f"  mean test success_rate = {per_seed['success_rate'].mean():.4f}")
    print(f"  std  test success_rate = {per_seed['success_rate'].std():.4f}")
    print(f"  mean test return       = {per_seed['return'].mean():.3f}")
    print(f"  std  test return       = {per_seed['return'].std():.3f}")

[per-seed test results — Stage E (a) crosscomp-1000 (β1=4.0, β2=0)]
seed  success_rate   return  safety_cost  time_s  path_efficiency
  42        0.9200 -39.2983      15.9130 81.1200           0.6350
  43        0.9500 -33.2838      14.9456 80.4110           0.6540
  44        0.7000 -62.6797      15.1101 79.2150           0.6175
  45        0.9200 -38.5383      14.9768 80.8610           0.6395
  46        0.9000 -45.1294      16.3583 82.2970           0.6179

[summary]
  mean test success_rate = 0.8780
  std  test success_rate = 0.1011
  mean test return       = -43.786
  std  test return       = 11.366


In [ ]:
# overview summary（critic_penalty / target_q 诊断）
import csv

overview_path = RESULTS_ROOT / "summaries" / "overview.csv"
if overview_path.exists():
    with overview_path.open(encoding="utf-8") as fp:
        reader = csv.DictReader(fp)
        for row in reader:
            print(f"[overview] dataset={row['dataset']} pair={row['pair']} num_seeds={row['num_seeds']}")
            print(f"  mean_test_success_rate    = {float(row['mean_test_success_rate']):.4f}")
            print(f"  std_test_success_rate     = {float(row['std_test_success_rate']):.4f}")
            print(f"  mean_test_return          = {float(row['mean_test_return']):.3f}")
            print(f"  mean_critic_penalty       = {float(row['mean_critic_penalty']):.4f}  (β2=0 时只是 logging 量)")
            print(f"  mean_target_q             = {float(row['mean_target_q']):.3f}")
            penalty_ratio = float(row['mean_critic_penalty_ratio'])
            print(f"  mean_critic_penalty_ratio = {penalty_ratio:.4f}  (β2=0 → β2·ratio = 0)")
else:
    print(f"[warn] overview.csv missing at {overview_path}")

[overview] dataset=crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000 pair=actorb_4p0__criticb_0p0 num_seeds=5
  mean_test_success_rate    = 0.8780
  std_test_success_rate     = 0.0904
  mean_test_return          = -43.786
  mean_critic_penalty       = 0.0792  (β2=0 时只是 logging 量)
  mean_target_q             = -0.128
  mean_critic_penalty_ratio = 0.1080  (β2=0 → β2·ratio = 0)


## 5. Cross-dataset Finding 1 二次验证：与 Stage C / worldcomp probe 三向对比

把 Stage E (a) 数字加入 Finding 1 主对照表：

1. **Δ vs Stage C 主 finalist**（同 dataset 同 seeds，仅 β2 不同）→ critic penalty 在 crosscomp 上对 mean 的贡献。
2. **Δ vs worldcomp probe**（同 β1=4.0, β2=0，跨 dataset）→ critic penalty 缺失的影响是否 dataset-invariant。
3. **mean_target_q 跳升幅度** vs worldcomp probe → 核心机制诊断（critic penalty 是否同样通过抑制 Q 高估起作用）。

In [ ]:
if not per_seed.empty:
    stage_e_mean = per_seed["success_rate"].mean()
    stage_e_std  = per_seed["success_rate"].std()
    stage_e_ret  = per_seed["return"].mean()

    # Stage C 主 finalist on crosscomp-1000（5 seeds, test=100）
    stage_c_mean = 0.902
    stage_c_std  = 0.021
    stage_c_target_q = -8.25  # from results/offline/rebrac/formal/summaries/overview.csv
    stage_c_ratio = 0.043     # β2·ratio for (β1=4.0, β2=2.0) on crosscomp-1000

    # Stage D worldcomp probe（β1=4.0, β2=0, 2 seeds 42/43, test=100）
    worldcomp_probe_mean = 0.910
    worldcomp_probe_std  = 0.014
    worldcomp_probe_target_q = 22.30
    worldcomp_phase1_target_q = 15.22

    # 当前实验 mean_target_q
    overview_path = RESULTS_ROOT / "summaries" / "overview.csv"
    stage_e_target_q = None
    if overview_path.exists():
        with overview_path.open(encoding="utf-8") as fp:
            reader = csv.DictReader(fp)
            for row in reader:
                if row["pair"] == PAIR_TAG and row["dataset"] == DATASET_NAME:
                    stage_e_target_q = float(row["mean_target_q"])
                    break

    print("=" * 84)
    print(f"{'Protocol':<58}{'mean':>8}{'std':>8}{'target_q':>10}")
    print("-" * 84)
    print(f"{'Stage C (β1=4.0, β2=2.0) crosscomp-1000 (5 seeds)':<58}{stage_c_mean:>8.3f}{stage_c_std:>8.3f}{stage_c_target_q:>10.2f}")
    print(f"{'Stage D probe (β1=4.0, β2=0) worldcomp-1000 (2 seeds)':<58}{worldcomp_probe_mean:>8.3f}{worldcomp_probe_std:>8.3f}{worldcomp_probe_target_q:>10.2f}")
    print(f"{'Stage E (a) (β1=4.0, β2=0) crosscomp-1000 (5 seeds)':<58}{stage_e_mean:>8.4f}{stage_e_std:>8.4f}{stage_e_target_q if stage_e_target_q is not None else float('nan'):>10.2f}")
    print("=" * 84)
    print()

    # 关键 Δ
    delta_vs_stagec = stage_e_mean - stage_c_mean
    print(f"Δ vs Stage C 主 finalist (同 dataset, β2=2 → β2=0): {delta_vs_stagec:+.4f} ({delta_vs_stagec*100:+.1f}pp)")
    print(f"  → critic penalty 在 crosscomp-1000 上对 mean success 的贡献")
    print()

    delta_vs_worldcomp_probe = stage_e_mean - worldcomp_probe_mean
    print(f"Δ vs worldcomp probe (β2=0, 跨 dataset): {delta_vs_worldcomp_probe:+.4f} ({delta_vs_worldcomp_probe*100:+.1f}pp)")
    print(f"  → critic penalty 缺失的影响是否 dataset-invariant")
    print()

    # mean_target_q 跳升对照
    if stage_e_target_q is not None:
        delta_target_q_crosscomp = stage_e_target_q - stage_c_target_q
        delta_target_q_worldcomp = worldcomp_probe_target_q - worldcomp_phase1_target_q
        print(f"mean_target_q 跳升对照（β2=0 vs β2=2 同 dataset）:")
        print(f"  crosscomp: {stage_c_target_q:.2f} → {stage_e_target_q:.2f}  Δ = {delta_target_q_crosscomp:+.2f}")
        print(f"  worldcomp: {worldcomp_phase1_target_q:.2f} → {worldcomp_probe_target_q:.2f}  Δ = {delta_target_q_worldcomp:+.2f} (+46%)")
        if abs(delta_target_q_crosscomp) >= 0.3 * abs(stage_c_target_q):
            print(f"  → Q 高估机制 dataset-invariant：critic penalty 在两个 dataset 上都通过 target Q bootstrap 抑制 Q 估计")
        else:
            print(f"  → Q 高估机制 dataset-dependent：crosscomp 上 critic penalty 通过其它路径起作用")

Protocol                                                      mean     std  target_q
------------------------------------------------------------------------------------
Stage C (β1=4.0, β2=2.0) crosscomp-1000 (5 seeds)            0.902   0.021     -8.25
Stage D probe (β1=4.0, β2=0) worldcomp-1000 (2 seeds)        0.910   0.014     22.30
Stage E (a) (β1=4.0, β2=0) crosscomp-1000 (5 seeds)         0.8780  0.1011     -0.13

Δ vs Stage C 主 finalist (同 dataset, β2=2 → β2=0): -0.0240 (-2.4pp)
  → critic penalty 在 crosscomp-1000 上对 mean success 的贡献

Δ vs worldcomp probe (β2=0, 跨 dataset): -0.0320 (-3.2pp)
  → critic penalty 缺失的影响是否 dataset-invariant

mean_target_q 跳升对照（β2=0 vs β2=2 同 dataset）:
  crosscomp: -8.25 → -0.13  Δ = +8.12
  worldcomp: 15.22 → 22.30  Δ = +7.08 (+46%)
  → Q 高估机制 dataset-invariant：critic penalty 在两个 dataset 上都通过 target Q bootstrap 抑制 Q 估计


## 6. 自动判定情形 A/B/C

按 §1 "判定区间" 自动给出 verdict。

In [ ]:
if not per_seed.empty:
    print("=" * 80)
    print("Stage E (a) 判定")
    print("-" * 80)
    print(f"  Stage E (a) mean_test_success = {stage_e_mean:.4f}")
    print(f"  Stage C 主 finalist baseline  = {stage_c_mean:.4f}")
    print(f"  Δ                             = {delta_vs_stagec:+.4f} ({delta_vs_stagec*100:+.1f}pp)")
    print()

    if stage_e_mean >= stage_c_mean - 0.02:
        verdict = (
            "情形 A：几乎一致（Stage C ± 2pp 内）\n"
            "  → dual penalty 在 crosscomp 上也几乎可省，Finding 1 跨 dataset 成立。\n"
            "  → Stage E 收口，无需 dual penalty 全因子 ablation。"
        )
    elif stage_e_mean >= 0.85:
        verdict = (
            "情形 B：略有下降（与 worldcomp probe 量级一致）\n"
            "  → critic penalty 对 mean 贡献小但非零，cross-dataset 一致。\n"
            "  → Finding 1 修正版（'小但非零 + Q 稳定性必要'）跨 dataset 成立；Stage E 收口。"
        )
    else:
        verdict = (
            "情形 C：显著下降\n"
            "  → crosscomp 上 critic penalty 对 mean 贡献远大于 worldcomp。\n"
            "  → Finding 1 需要在 dataset 维度上分别表述，Stage E 不能仅凭此实验收口。"
        )

    print("[Stage E (a) verdict]")
    print(verdict)
    print("=" * 80)

Stage E (a) 判定
--------------------------------------------------------------------------------
  Stage E (a) mean_test_success = 0.8780
  Stage C 主 finalist baseline  = 0.9020
  Δ                             = -0.0240 (-2.4pp)

[Stage E (a) verdict]
情形 B：略有下降（与 worldcomp probe 量级一致）
  → critic penalty 对 mean 贡献小但非零，cross-dataset 一致。
  → Finding 1 修正版（'小但非零 + Q 稳定性必要'）跨 dataset 成立；Stage E 收口。


## 7. seed 44 在 crosscomp 上的诊断对照

Stage C `(β1=4.0, β2=2.0)` 上 seed 44 是 0.870/0.890（已被 β1=4.0 完整回收）。Stage D Phase 1 deployable 上 seed 44 掉到 0.78。这两个 dataset 上 seed 44 的行为分叉（report §7.10.6 Finding 2）。Stage E (a) 在 crosscomp 上把 β2 也去掉，看 seed 44 是不是仍然被 β1 单独稳住——这是 cross-dataset Finding 2 的间接验证。

判据：
- seed 44 在 Stage E (a) 上 ≥ 0.85 → β1 在 crosscomp 上独立稳住 seed 44（critic penalty 不是 seed 44 在 crosscomp 上回收的必需）；
- seed 44 < 0.85 → critic penalty 在 crosscomp 上对 seed 44 也是必需，与 worldcomp Phase 2 假设 A 一致。

In [ ]:
if not per_seed.empty:
    seed_44_row = per_seed[per_seed["seed"] == "44"]
    if seed_44_row.empty:
        print("[warn] seed 44 缺失")
    else:
        s44_e = float(seed_44_row.iloc[0]["success_rate"])
        s44_stage_c = 0.880  # Stage C crosscomp-1000 (β1=4.0, β2=2.0) seed 44 (取 0.870/0.890 的中点)
        s44_phase1  = 0.780  # Stage D Phase 1 worldcomp deployable seed 44
        s44_phase2  = 0.900  # Stage D Phase 2 worldcomp privileged seed 44

        print("=" * 70)
        print("seed 44 跨 (dataset, β2) 对比")
        print("-" * 70)
        print(f"  Stage C crosscomp-1000 (β2=2):  {s44_stage_c:.3f}")
        print(f"  Stage E crosscomp-1000 (β2=0):  {s44_e:.3f}")
        print(f"  Stage D Phase 1 worldcomp dep (β2=2): {s44_phase1:.3f}")
        print(f"  Stage D Phase 2 worldcomp priv (β2=2): {s44_phase2:.3f}")
        print("-" * 70)

        delta_s44 = s44_e - s44_stage_c
        print(f"Δ seed 44 (crosscomp β2=0 vs β2=2): {delta_s44:+.3f}")

        if s44_e >= 0.85:
            s44_verdict = "β1=4.0 在 crosscomp 上独立稳住 seed 44 → critic penalty 在 crosscomp 上对 seed 44 不是必需"
        else:
            s44_verdict = "critic penalty 在 crosscomp 上对 seed 44 也是必需 → 与 worldcomp Phase 2 假设 A 一致（critic 信息瓶颈跨 dataset 成立）"
        print(f"\n[seed 44 verdict] {s44_verdict}")
        print("=" * 70)

seed 44 跨 (dataset, β2) 对比
----------------------------------------------------------------------
  Stage C crosscomp-1000 (β2=2):  0.880
  Stage E crosscomp-1000 (β2=0):  0.700
  Stage D Phase 1 worldcomp dep (β2=2): 0.780
  Stage D Phase 2 worldcomp priv (β2=2): 0.900
----------------------------------------------------------------------
Δ seed 44 (crosscomp β2=0 vs β2=2): -0.180

[seed 44 verdict] critic penalty 在 crosscomp 上对 seed 44 也是必需 → 与 worldcomp Phase 2 假设 A 一致（critic 信息瓶颈跨 dataset 成立）


## 8. 报告写入清单

跑完上面所有 cell 后：

1. **更新 [docs/rebrac_experiment_report.md](../docs/rebrac_experiment_report.md)**：
   - 新增 §7.14 「Stage E (a)：crosscomp-1000 critic-penalty-off cross-dataset 二次验证」小节，含 per-seed 表、与 Stage C / worldcomp probe 三向对比、mean_target_q 对照、seed 44 跨 dataset 诊断、verdict；
   - §1 一句话当前状态更新为 "Stage E 收口完成"；
   - §10 最终结论补一条 Stage E 结论；
   - §11 文件索引追加 Stage E 路径。
2. **更新 [docs/rebrac_experiment_plan.md](../docs/rebrac_experiment_plan.md)**：
   - §6.6 Stage E 标记为 `【已完成】`；
   - §10 推荐执行顺序更新到 step 7 完成；
   - 头部 rev.5 → rev.6。
3. **如果触发情形 C**（mean < 0.85）：Finding 1 不能跨 dataset 简单 generalize；需要在 §7.13 Finding 1 修正版基础上再加一个 dataset 维度的 caveat。